# Feature Engineering (2025)

Build model features and targets from the preprocessed pitch table.

**Input:** `data/preprocessed_2025.parquet` (`02_preprocessing.ipynb`)

**Output:** `data/modeling_frame_2025.parquet`

Logic lives in `src/feature_engineering.py`.

## Design notes (v2)

- **No `batter_te`.** Models learn from continuous **season-to-date rolling hitter metrics**, not target-encoded batter IDs. This supports zero-shot inference for unseen hitters (app roster, new call-ups).
- **Rolling metrics are leak-safe.** Each row uses only *prior* pitches for that batter (`shift(1)` within batter, sorted by date / game / AB / pitch). First pitch of a batter's sample gets league-default fill values.
- **Platoon + 3D release** features are engineered here and evaluated in RF forward selection (`same_handed`, release point, extension, movement).

## Features

### Location & movement (numeric)

| Column | Definition |
|--------|------------|
| `plate_x`, `plate_z` | Plate location (ft) |
| `miss_dist_in` | Inches from pitch to nearest rule-book zone edge (0 if inside) |
| `center_dist_in` | Inches from pitch to zone center (`plate_x=0`, `z=(sz_top+sz_bot)/2`) |
| `norm_hb` | Horizontal break; LHP `pfx_x` × −1 |
| `norm_ivb` | Induced vertical break (`pfx_z`) |
| `release_speed`, `release_spin_rate` | Pitch velocity / spin |
| `release_pos_x`, `release_pos_z`, `release_extension` | 3D release point & extension (Statcast) |

### Rolling batter profile (numeric) — replaces `batter_te`

Computed **before each pitch** from that batter's prior pitches in the season:

| Column | Definition |
|--------|------------|
| `swing_pct` | Prior swings ÷ prior pitches seen |
| `whiff_pct` | Prior whiffs ÷ prior swings |
| `z_swing_pct` | Prior swings on in-zone pitches ÷ prior in-zone pitches |
| `o_swing_pct` | Prior swings on out-of-zone pitches ÷ prior out-of-zone pitches |
| `z_whiff_pct` | Prior in-zone whiffs ÷ prior in-zone swings (Z-Zone Whiff%) |
| `o_whiff_pct` | Prior out-of-zone whiffs ÷ prior out-of-zone swings (O-Zone Whiff%) |

Cold-start rows (no history) are filled with league defaults from the full frame.

### Rolling bat-tracking profile (numeric)

Season-to-date rolling metrics from **prior tracked swings** (Statcast `bat_speed` / `attack_angle` when available in the pull):

| Column | Definition |
|--------|------------|
| `bat_speed` | Mean bat speed (mph) on prior swings with tracking |
| `attack_angle` | Mean attack angle (deg) on prior tracked swings |
| `squared_up_rate` | Share of prior tracked swings meeting squared-up criteria |

Used by **whiff** (07) and **xwOBAcon** (09) models — **not** the swing model (05). If raw tracking columns are missing from `preprocessed_2025.parquet`, rows are filled with league defaults until you re-pull Statcast (schema v4).

### Binary

| Column | Definition |
|--------|------------|
| `is_in_zone` | **1** if `-0.708 ≤ plate_x ≤ +0.708` and `sz_bot ≤ plate_z ≤ sz_top`; else **0** |
| `same_handed` | **1** if `p_throws == stand` (same-side platoon); else **0** |
| `p_throws_L` | **1** if pitcher is LHP; else **0** (in `pitcher_delivery` group) |

### Pitch physics (three candidate groups)

| Group | Columns |
|-------|----------|
| `pitch_movement` | `norm_hb`, `norm_ivb` |
| `pitcher_delivery` | `release_pos_x`, `release_pos_y`, `release_pos_z`, `release_extension`, `p_throws_L` |
| `pitch_quality` | `release_speed`, `release_spin_rate` |

### Categorical (one-hot)

**Base state:** `state_empty`, `state_1b`, `state_1b_2b`, `state_1b_3b`, `state_2b`, `state_2b_3b`, `state_3b`, `state_loaded`

**Count:** `count_0_0` … `count_3_2`

**Attack zone:** `zone_heart`, `zone_shadow`, `zone_chase`, `zone_waste`

**Pitch type:** `pitch_FF`, `pitch_SL`, … (10 competitive types)

## Targets

| Column | Definition |
|--------|------------|
| `is_swing` | 1 if full swing (whiffs, fouls, BIP); **not** bunts (`foul_bunt`, `missed_bunt`) |
| `is_whiff` | 1 if `swinging_strike` or `swinging_strike_blocked` (swings only) |
| `xwobacon` | `estimated_woba_using_speedangle` on BIP only; **NaN** otherwise |

## Downstream split (modeling notebooks 05 / 07 / 09)

This notebook builds the **full-season** modeling frame. Chronological splits happen later:

| Split | Dates | Purpose |
|-------|-------|---------|
| Train | Mar–Jul 2025 | Fit RF models |
| Val | Aug 2025 | Forward feature-group selection |
| Test | Sep 2025 | Locked final holdout |

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

NB_DIR = Path.cwd().resolve()
ROOT = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import importlib
import src.feature_engineering as feature_engineering

importlib.reload(feature_engineering)
CATEGORICAL_FEATURE_COLUMNS = feature_engineering.CATEGORICAL_FEATURE_COLUMNS
MODELING_FEATURE_COLUMNS = feature_engineering.MODELING_FEATURE_COLUMNS
NUMERIC_FEATURE_COLUMNS = feature_engineering.NUMERIC_FEATURE_COLUMNS
TARGET_COLUMNS = feature_engineering.TARGET_COLUMNS
build_modeling_frame = feature_engineering.build_modeling_frame

INPUT_FILE = ROOT / "data" / "preprocessed_2025.parquet"
OUTPUT_FILE = ROOT / "data" / "modeling_frame_2025.parquet"

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Run 02_preprocessing.ipynb first — missing {INPUT_FILE}")

preprocessed = pd.read_parquet(INPUT_FILE)
modeling_frame, report = build_modeling_frame(preprocessed)

print("Feature engineering summary")
print(f"• Input pitches: {report.n_input:,}")
print(f"• Unknown attack zone dropped: {report.n_unknown_attack_zone:,}")
print(f"• Final modeling frame: {report.n_final:,}")
print(f"• League default swing_pct: {report.league_swing_pct:.1%}")
print(f"• League default whiff_pct: {report.league_whiff_pct:.1%}")
print(f"• League default z_swing_pct: {report.league_z_swing_pct:.1%}")
print(f"• League default o_swing_pct: {report.league_o_swing_pct:.1%}")
print(f"• League default z_whiff_pct: {report.league_z_whiff_pct:.1%}")
print(f"• League default o_whiff_pct: {report.league_o_whiff_pct:.1%}")
print(f"• League default bat_speed: {report.league_bat_speed:.1f} mph")
print(f"• League default attack_angle: {report.league_attack_angle:.1f} deg")
print(f"• League default squared_up_rate: {report.league_squared_up_rate:.1%}")

raw_tracking = [c for c in ("bat_speed", "attack_angle") if c in preprocessed.columns]
if raw_tracking:
    tracked = preprocessed["bat_speed"].notna().sum() if "bat_speed" in preprocessed.columns else 0
    print(f"• Raw bat-tracking columns in preprocessed: {raw_tracking} ({tracked:,} non-null bat_speed rows)")
else:
    print("• Raw bat-tracking columns missing — re-run 00_data_pull with schema v4, then 02_preprocessing")

modeling_frame.to_parquet(OUTPUT_FILE, index=False)
print(f"\nSaved → {OUTPUT_FILE}")

Feature engineering summary
• Input pitches: 695,622
• Unknown attack zone dropped: 0
• Final modeling frame: 695,622
• League default swing_pct: 47.6%
• League default whiff_pct: 23.2%
• League default z_swing_pct: 69.0%
• League default o_swing_pct: 32.3%
• League default z_whiff_pct: 13.8%
• League default o_whiff_pct: 37.4%
• League default bat_speed: 69.9 mph
• League default attack_angle: 9.2 deg
• League default squared_up_rate: 1.7%
• Raw bat-tracking columns in preprocessed: ['bat_speed', 'attack_angle'] (322,993 non-null bat_speed rows)

Saved → C:\Users\tabshire\Desktop\Portfolio\data\modeling_frame_2025.parquet


In [ ]:
BATTER_ROLLING_COLUMNS = feature_engineering.BATTER_ROLLING_COLUMNS
BAT_TRACKING_COLUMNS = feature_engineering.BAT_TRACKING_COLUMNS
PLATOON_COLUMNS = feature_engineering.PLATOON_COLUMNS
PITCH_MOVEMENT_COLUMNS = feature_engineering.PITCH_MOVEMENT_COLUMNS
PITCHER_DELIVERY_COLUMNS = feature_engineering.PITCHER_DELIVERY_COLUMNS
PITCH_QUALITY_COLUMNS = feature_engineering.PITCH_QUALITY_COLUMNS
PHYSICS_COLUMNS = feature_engineering.PHYSICS_COLUMNS

print("Feature columns")
print("  Location baseline:", feature_engineering.BASELINE_FEATURES)
print("  Raw plate coords:", ["plate_x", "plate_z"])
print("  Rolling batter:", BATTER_ROLLING_COLUMNS)
print("  Bat tracking (rolling):", BAT_TRACKING_COLUMNS)
print("  Platoon:", PLATOON_COLUMNS)
print("  Pitch movement:", PITCH_MOVEMENT_COLUMNS)
print("  Pitcher delivery:", PITCHER_DELIVERY_COLUMNS)
print("  Pitch quality:", PITCH_QUALITY_COLUMNS)
print(f"  Categorical one-hot: {len(CATEGORICAL_FEATURE_COLUMNS)} columns")
print(f"  Total modeling features: {len(MODELING_FEATURE_COLUMNS)}")
print("  Targets:", TARGET_COLUMNS)

missing_bt = [c for c in BAT_TRACKING_COLUMNS if c not in modeling_frame.columns]
if missing_bt:
    raise RuntimeError(f"Modeling frame missing bat-tracking columns: {missing_bt}")

print("\nTarget rates")
print(f"  Swing rate: {modeling_frame['is_swing'].mean():.1%}")
print(f"  Whiff rate (swings): {modeling_frame.loc[modeling_frame['is_swing']==1, 'is_whiff'].mean():.1%}")
print(
    f"  xwOBAcon mean (contact): {modeling_frame['xwobacon'].dropna().mean():.3f} "
    f"({modeling_frame['xwobacon'].notna().sum():,} BIP rows)"
)

print("\nRolling metric snapshot (should vary by batter history)")
display(
    modeling_frame.groupby("batter")[BATTER_ROLLING_COLUMNS]
    .mean()
    .sort_values("swing_pct")
    .head(5)
)

print("\nBat-tracking snapshot (rolling; flat = league default fill until re-pull)")
display(
    modeling_frame.groupby("batter")[BAT_TRACKING_COLUMNS]
    .mean()
    .sort_values("bat_speed")
    .head(5)
)

display(modeling_frame[MODELING_FEATURE_COLUMNS + TARGET_COLUMNS].head())

Feature columns
  Location baseline: ['is_in_zone', 'miss_dist_in', 'center_dist_in']
  Raw plate coords: ['plate_x', 'plate_z']
  Rolling batter: ['swing_pct', 'whiff_pct', 'z_swing_pct', 'o_swing_pct', 'z_whiff_pct', 'o_whiff_pct']
  Bat tracking (rolling): ['bat_speed', 'attack_angle', 'squared_up_rate']
  Platoon: ['same_handed']
  Pitch movement: ['norm_hb', 'norm_ivb']
  Pitcher delivery: ['release_pos_x', 'release_pos_y', 'release_pos_z', 'release_extension', 'p_throws_L']
  Pitch quality: ['release_speed', 'release_spin_rate']
  Categorical one-hot: 34 columns
  Total modeling features: 58
  Targets: ['is_swing', 'is_whiff', 'xwobacon']

Target rates
  Swing rate: 47.6%
  Whiff rate (swings): 23.2%
  xwOBAcon mean (contact): 0.369 (121,159 BIP rows)

Rolling metric snapshot (should vary by batter history)


,swing_pct,whiff_pct,z_swing_pct,o_swing_pct,z_whiff_pct,o_whiff_pct
batter,,,,,,
669194,0.118969,0.231586,0.344818,0.080744,0.138448,0.373841
829272,0.158626,0.231586,0.229879,0.322974,0.138448,0.373841
683004,0.158626,0.231586,0.229879,0.322974,0.138448,0.373841
675915,0.210973,0.574126,0.767227,0.163288,0.353836,0.624421
681909,0.264293,0.569061,0.274815,0.271060,0.451740,0.545033



Bat-tracking snapshot (rolling; flat = league default fill until re-pull)


,bat_speed,attack_angle,squared_up_rate
batter,,,
650333,60.585789,4.775543,0.111866
802415,61.191551,2.851517,0.131756
669397,61.278412,5.500925,0.180108
680757,61.562054,1.780694,0.060152
805779,61.833858,0.924186,0.083848


,plate_x,plate_z,miss_dist_in,center_dist_in,norm_hb,norm_ivb,release_pos_x,release_pos_y,release_pos_z,release_extension,...,pitch_FF,pitch_FS,pitch_KC,pitch_SI,pitch_SL,pitch_ST,pitch_SV,is_swing,is_whiff,xwobacon
0,0.173717,1.205657,5.766714,16.786634,-0.57,1.39,-2.64,54.400002,5.84,6.1,...,1,0,0,0,0,0,0,0,0,NaN
1,0.344554,1.666801,0.000000,9.884674,-0.05,0.22,-2.63,54.509998,5.83,6.0,...,0,0,0,0,1,0,0,1,1,NaN
2,0.795760,0.248935,16.732025,29.713882,0.06,-0.01,-2.72,54.509998,5.77,6.0,...,0,0,0,0,1,0,0,0,0,NaN
3,-0.739459,2.893887,0.373508,9.796345,-0.68,1.40,-2.55,54.480000,5.93,6.0,...,1,0,0,0,0,0,0,0,0,NaN
4,1.694437,-0.434296,26.482355,39.780685,0.46,-0.51,-2.52,54.380001,5.73,6.1,...,0,0,0,0,1,0,0,1,1,NaN
